## Finding Second Highest Salary by Department

This notebook demonstrates three different approaches to finding the second highest salary in each department using PySpark window functions. It's an excellent comparison of how **row_number()**, **rank()**, and **dense_rank()** handle tied values differently.

### Sample Data
The dataset contains 6 employees across Sales and Engineering departments, with Bob and Charlie both earning $60,000 (a tie in the Sales department).

### Three Approaches Compared

**1. row_number()** - Assigns sequential numbers even for ties. Bob gets rank 1, Charlie gets rank 2 despite having identical salaries. Returns David (Engineering, $80K) and Charlie (Sales, $60K), but Charlie wasn't truly the second highest—he tied for first.

**2. rank()** - Assigns the same rank to ties but skips subsequent ranks. Bob and Charlie both get rank 1, then Alice jumps to rank 3 (skipping rank 2). Only returns David (Engineering, $80K) because no one in Sales has rank 2 due to the gap created by the tie.

**3. dense_rank()** - Assigns the same rank to ties without gaps. Bob and Charlie both get rank 1, then Alice gets rank 2. Returns David (Engineering, $80K) and Alice (Sales, $50K)—the truly second-highest distinct salaries.

### Key Insight
For finding the "nth highest" value, `dense_rank()` is typically the correct choice because it properly handles ties and counts distinct salary levels, not individual rows.

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, dense_rank,row_number,rank

data = [
    ("Alice", "Sales", 50000),
    ("Bob", "Sales", 60000),
    ("Charlie", "Sales", 60000),
    ("David", "Engineering", 80000),
    ("Eve", "Engineering", 90000),
    ("Frank", "Engineering", 75000)
]

columns = ["employee_name", "department", "salary"]
df = spark.createDataFrame(data, columns)
df.show()

+-------------+-----------+------+
|employee_name| department|salary|
+-------------+-----------+------+
|        Alice|      Sales| 50000|
|          Bob|      Sales| 60000|
|      Charlie|      Sales| 60000|
|        David|Engineering| 80000|
|          Eve|Engineering| 90000|
|        Frank|Engineering| 75000|
+-------------+-----------+------+



In [0]:
# 1. Define the Window Specification
# We partition by department, and order by salary descending
windowSpec = Window.partitionBy("department").orderBy(col("salary").desc())

# 2. Apply the dense_rank function
ranked_df = df.withColumn("rank", row_number().over(windowSpec))
ranked_df.show()
# 3. Filter for the second rank and drop the rank column for clean output
second_highest_df = ranked_df.filter(col("rank") == 2).drop("rank")

second_highest_df.show()

+-------------+-----------+------+----+
|employee_name| department|salary|rank|
+-------------+-----------+------+----+
|          Eve|Engineering| 90000|   1|
|        David|Engineering| 80000|   2|
|        Frank|Engineering| 75000|   3|
|          Bob|      Sales| 60000|   1|
|      Charlie|      Sales| 60000|   2|
|        Alice|      Sales| 50000|   3|
+-------------+-----------+------+----+

+-------------+-----------+------+
|employee_name| department|salary|
+-------------+-----------+------+
|        David|Engineering| 80000|
|      Charlie|      Sales| 60000|
+-------------+-----------+------+



In [0]:
# 1. Define the Window Specification
# We partition by department, and order by salary descending
windowSpec = Window.partitionBy("department").orderBy(col("salary").desc())

# 2. Apply the dense_rank function
ranked_df = df.withColumn("rank", rank().over(windowSpec))
ranked_df.show()
# 3. Filter for the second rank and drop the rank column for clean output
second_highest_df = ranked_df.filter(col("rank") == 2).drop("rank")

second_highest_df.show()

+-------------+-----------+------+----+
|employee_name| department|salary|rank|
+-------------+-----------+------+----+
|          Eve|Engineering| 90000|   1|
|        David|Engineering| 80000|   2|
|        Frank|Engineering| 75000|   3|
|          Bob|      Sales| 60000|   1|
|      Charlie|      Sales| 60000|   1|
|        Alice|      Sales| 50000|   3|
+-------------+-----------+------+----+

+-------------+-----------+------+
|employee_name| department|salary|
+-------------+-----------+------+
|        David|Engineering| 80000|
+-------------+-----------+------+



In [0]:
# 1. Define the Window Specification
# We partition by department, and order by salary descending
windowSpec = Window.partitionBy("department").orderBy(col("salary").desc())

# 2. Apply the dense_rank function
ranked_df = df.withColumn("rank", dense_rank().over(windowSpec))
ranked_df.show()
# 3. Filter for the second rank and drop the rank column for clean output
second_highest_df = ranked_df.filter(col("rank") == 2).drop("rank")

second_highest_df.show()

+-------------+-----------+------+----+
|employee_name| department|salary|rank|
+-------------+-----------+------+----+
|          Eve|Engineering| 90000|   1|
|        David|Engineering| 80000|   2|
|        Frank|Engineering| 75000|   3|
|          Bob|      Sales| 60000|   1|
|      Charlie|      Sales| 60000|   1|
|        Alice|      Sales| 50000|   2|
+-------------+-----------+------+----+

+-------------+-----------+------+
|employee_name| department|salary|
+-------------+-----------+------+
|        David|Engineering| 80000|
|        Alice|      Sales| 50000|
+-------------+-----------+------+

